# Factor construction

This notebook turns the raw data collected in `01_data_collection.ipynb` into the daily factor files in `factors/`. It needs the licensed raw files in `data/private/`, which are not published, so it does not run from a fresh clone. Its output files are published.

In [1]:
# This notebook needs the raw financial statements and market values, which are licensed and not published
import os

if not (os.path.exists('data/private/financials.csv.gz') and os.path.exists('data/private/market_value.csv.gz')):
    raise SystemExit('This notebook needs the licensed raw files in data/private/. The factors it produces are already in factors/, and the analysis notebooks (03 to 05) run with them.')

In [2]:
# Library imports

import pandas as pd
from datetime import date, datetime
from workadays import workdays as wd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from  bizdays import Calendar, set_option
from time import sleep

cal = Calendar.load('ANBIMA')
set_option('mode', 'pandas')

In [3]:
# Imports data already collected to make the analysis easier

path = '.'
factors_folder = 'factors'
data_folder = 'data'

companies = pd.read_csv(path + "/" + data_folder + "/" + 'index_composition.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')
financials = pd.read_csv(path + "/" + data_folder + "/" + 'private/financials.csv.gz')
prices_df = pd.read_csv(path + "/" + data_folder + "/" + 'prices.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')
market_value = pd.read_csv(path + "/" + data_folder + "/" + 'private/market_value.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')
volume = pd.read_csv(path + "/" + data_folder + "/" + 'volume.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')
benchmarks = pd.read_csv(path + "/" + data_folder + "/" + 'benchmarks.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')

# Index rebalancing dates
rebalance_dates = sorted(companies.index.unique())

# Backtest dates
ini_date = rebalance_dates[0]
end_date = date.today()

financials['statement_date'] = pd.to_datetime(financials['statement_date'], format='%Y-%m-%d')
financials['publication_date'] = pd.to_datetime(financials['publication_date'], format='%Y-%m-%d')
financials['collection_date'] = pd.to_datetime(financials['collection_date'], format='%Y-%m-%d')

### Indicator matrix

The next step is to calculate the indicators for all days of the backtest; at the end we have one table per factor, with dates up to 2026 and companies as columns.

The idea is that the portfolio can be built on any date within the backtest period.

### Factor definitions

Accounting values enter each series on their publication date (point-in-time), and flow items are the sum of the last four quarters (LTM). Lagged means four quarters earlier. Files are saved in `factors/`, with raw values, one row per day and one column per ticker.

| Family | File | Definition |
|---|---|---|
| Value | `earnings_yield` | LTM net income (if positive) / market value |
| Value | `earnings_yield_mf` | LTM EBIT (if positive) / (market value + net debt) |
| Value | `book_to_price` | Equity / market value |
| Value | `sales_to_price` | LTM net revenue (if positive) / market value |
| Value | `fcf_to_price` | Free cash flow (if positive) / market value |
| Value | `ebitda_ev` | LTM EBITDA (if positive) / (market value + net debt) |
| Profitability | `roe` | LTM net income / lagged equity |
| Profitability | `roa` | LTM EBIT / lagged total assets |
| Profitability | `gpoa` | LTM gross profit / lagged total assets |
| Profitability | `cfoa` | Free cash flow / lagged total assets, with free cash flow = LTM operating cash flow − change in PP&E + LTM D&A |
| Profitability | `acc` | −Change in net operating assets / lagged total assets |
| Profitability | `roc_mf` | LTM EBIT / lagged (net working capital + tangible assets) |
| Growth | `g_revenue`, `g_earnings`, `g_ebitda` | Year-on-year growth of the LTM value, when the base is positive |
| Growth | `investments` | −Year-on-year growth of total assets |
| Leverage | `debt_to_equity`, `debt_to_assets` | Net debt / equity, net debt / total assets |
| Leverage | `debt_to_ebitda` | Net debt / (LTM EBIT + LTM D&A), when positive |
| Leverage | `liab_to_equity` | Total liabilities / equity |
| Leverage | `debt_ebit_mf` | Net debt / LTM EBIT, when positive |
| Price | `momentum` | P(t−21) / P(t−252) − 1 |
| Price | `reversal` | −(P(t) / P(t−21) − 1) |
| Price | `volatility` | 231-day standard deviation of daily returns, annualized and lagged 21 days |
| Price | `bab` | −(0.6 · ρ · σ_stock / σ_market + 0.4), with ρ over 5 years of 3-day returns and σ over 1 year of daily returns (Frazzini & Pedersen, 2014) |

The code of these calculations is in `build_indicators` (`brfactors/indicators.py`) and in the cells below. `ivol` and `ff_factors` come from `03_historical_betas.ipynb`.

In [4]:
from brfactors.indicators import pit_matrix, build_indicators

In [5]:
# financials = concat.copy()

# Calculates the 'noa' and 'net_debt' columns directly
financials['noa'] = financials['total_assets'] - financials['cash'] - (financials['current_liabilities'] + financials['non_current_liabilities']) + financials['st_debt'].fillna(0) + financials['lt_debt'].fillna(0)
financials['net_debt'] = financials['st_debt'].fillna(0) + financials['lt_debt'].fillna(0) - financials['cash']
financials['nwc'] = financials['current_assets'] - financials['current_liabilities'] + financials['st_debt'].fillna(0)
financials['tangible_assets'] = financials['total_assets'] - financials['current_assets'] - financials['intangibles'].fillna(0)

# Clears negative values in equity and total assets
financials['equity'] = financials['equity'].where(financials['equity'] > 0, np.nan)
financials['total_assets'] = financials['total_assets'].where(financials['total_assets'] > 0, np.nan)

# Calculates the indicators for all companies and concatenates them in the dataframe
indicators = pd.concat([build_indicators(financials, stk, prices_df.index) for stk in tqdm(financials['ticker'].unique())])

In [6]:
# Uses pivot_table to create a dataframe with a single indicator
# Profitability
roe = indicators.reset_index().pivot(index = 'Date', columns = 'ticker', values = 'roe').dropna(how = 'all')
roa = indicators.reset_index().pivot(index = 'Date', columns = 'ticker', values = 'roa').dropna(how = 'all')
gpoa = indicators.reset_index().pivot(index = 'Date', columns = 'ticker', values = 'gpoa').dropna(how = 'all')
cfoa = indicators.reset_index().pivot(index = 'Date', columns = 'ticker', values = 'cfoa').dropna(how = 'all')
acc = indicators.reset_index().pivot(index = 'Date', columns = 'ticker', values = 'acc').dropna(how = 'all')

# Growth
g_revenue = indicators.reset_index().pivot(index = 'Date', columns = 'ticker', values = 'g_revenue').dropna(how = 'all')
g_ebitda = indicators.reset_index().pivot(index = 'Date', columns = 'ticker', values = 'g_ebitda').dropna(how = 'all')
g_earnings = indicators.reset_index().pivot(index = 'Date', columns = 'ticker', values = 'g_earnings').dropna(how = 'all')

# Leverage
debt_to_equity = indicators.reset_index().pivot(index = 'Date', columns = 'ticker', values = 'debt_to_equity').dropna(how = 'all')
debt_to_assets = indicators.reset_index().pivot(index = 'Date', columns = 'ticker', values = 'debt_to_assets').dropna(how = 'all')
debt_to_ebitda = indicators.reset_index().pivot(index = 'Date', columns = 'ticker', values = 'debt_to_ebitda').dropna(how = 'all')
liab_to_equity = indicators.reset_index().pivot(index = 'Date', columns = 'ticker', values = 'liab_to_equity').dropna(how = 'all')

# Investments
investments = indicators.reset_index().pivot(index = 'Date', columns = 'ticker', values = 'investments').dropna(how = 'all')

# Magic Formula
roc_mf = indicators.reset_index().pivot(index = 'Date', columns = 'ticker', values = 'roc_mf').dropna(how = 'all')
debt_ebit_mf = indicators.reset_index().pivot(index = 'Date', columns = 'ticker', values = 'debt_ebit_mf').dropna(how = 'all')

In [7]:
# Creates a daily series with the indicator of all companies
net_revenue = indicators.reset_index().pivot(index = 'Date', columns = 'ticker', values = 'net_revenue')
ebit = indicators.reset_index().pivot(index = 'Date', columns = 'ticker', values = 'ebit')
ebitda = indicators.reset_index().pivot(index = 'Date', columns = 'ticker', values = 'ebitda')
net_income = indicators.reset_index().pivot(index = 'Date', columns = 'ticker', values = 'net_income')
equity = indicators.reset_index().pivot(index = 'Date', columns = 'ticker', values = 'equity')
net_debt = indicators.reset_index().pivot(index = 'Date', columns = 'ticker', values = 'net_debt')
fcf = indicators.reset_index().pivot(index = 'Date', columns = 'ticker', values = 'fcf')

# Value
earnings_yield = net_income.map(lambda x: x if x > 0 else np.nan) / market_value
earnings_yield_mf = ebit.map(lambda x: x if x > 0 else np.nan) / (market_value + net_debt)
book_to_price = equity / market_value
sales_to_price = net_revenue.map(lambda x: x if x > 0 else np.nan) / market_value
fcf_to_price = fcf.map(lambda x: x if x > 0 else np.nan) / market_value
ebitda_ev = ebitda.map(lambda x: x if x > 0 else np.nan) / (market_value + net_debt)

In [8]:
momentum = (prices_df.shift(21) / prices_df.shift(252)).sub(1)
reversal = -(prices_df / prices_df.shift(21)).sub(1)
volatility = prices_df.pct_change(fill_method = None).rolling(252-21).std().mul(np.sqrt(252)).shift(21)

In [9]:
# Betting Against Beta (Frazzini & Pedersen, 2014)
# beta = corr(5 years, 3-day returns) * vol_stock / vol_market (1 year, daily returns)
log_ret = np.log(prices_df).diff()
log_mkt = np.log1p(benchmarks['IBRX']).reindex(log_ret.index)

vol_stock = log_ret.rolling(252, min_periods=120).std()
vol_mkt = log_mkt.rolling(252, min_periods=120).std()

ret_3d = log_ret.rolling(3).sum()
mkt_3d = log_mkt.rolling(3).sum()
corr = ret_3d.rolling(1260, min_periods=750).corr(mkt_3d)

# Shrinkage towards the cross-sectional beta (w = 0.6)
beta = 0.6 * corr.mul(vol_stock.div(vol_mkt, axis=0)) + 0.4

# BAB signal: inverted beta (higher = better), as in reversal
bab = -beta

In [10]:
# Saves the factor dataframes to files
roe.to_csv(path + "/" + factors_folder + "/" + 'roe.csv.gz')
roa.to_csv(path + "/" + factors_folder + "/" + 'roa.csv.gz')
gpoa.to_csv(path + "/" + factors_folder + "/" + 'gpoa.csv.gz')
cfoa.to_csv(path + "/" + factors_folder + "/" + 'cfoa.csv.gz')
acc.to_csv(path + "/" + factors_folder + "/" + 'acc.csv.gz')

g_revenue.to_csv(path + "/" + factors_folder + "/" + 'g_revenue.csv.gz')
g_ebitda.to_csv(path + "/" + factors_folder + "/" + 'g_ebitda.csv.gz')
g_earnings.to_csv(path + "/" + factors_folder + "/" + 'g_earnings.csv.gz')

debt_to_equity.to_csv(path + "/" + factors_folder + "/" + 'debt_to_equity.csv.gz')
debt_to_assets.to_csv(path + "/" + factors_folder + "/" + 'debt_to_assets.csv.gz')
debt_to_ebitda.to_csv(path + "/" + factors_folder + "/" + 'debt_to_ebitda.csv.gz')
liab_to_equity.to_csv(path + "/" + factors_folder + "/" + 'liab_to_equity.csv.gz')

earnings_yield.to_csv(path + "/" + factors_folder + "/" + 'earnings_yield.csv.gz')
book_to_price.to_csv(path + "/" + factors_folder + "/" + 'book_to_price.csv.gz')
sales_to_price.to_csv(path + "/" + factors_folder + "/" + 'sales_to_price.csv.gz')
fcf_to_price.to_csv(path + "/" + factors_folder + "/" + 'fcf_to_price.csv.gz')
ebitda_ev.to_csv(path + "/" + factors_folder + "/" + 'ebitda_ev.csv.gz')

roc_mf.to_csv(path + "/" + factors_folder + "/" + 'roc_mf.csv.gz')
earnings_yield_mf.to_csv(path + "/" + factors_folder + "/" + 'earnings_yield_mf.csv.gz')
debt_ebit_mf.to_csv(path + "/" + factors_folder + "/" + 'debt_ebit_mf.csv.gz')

investments.to_csv(path + "/" + factors_folder + "/" + 'investments.csv.gz')

momentum.to_csv(path + "/" + factors_folder + "/" + 'momentum.csv.gz')
reversal.to_csv(path + "/" + factors_folder + "/" + 'reversal.csv.gz')
volatility.to_csv(path + "/" + factors_folder + "/" + 'volatility.csv.gz')
bab.to_csv(path + "/" + factors_folder + "/" + 'bab.csv.gz')

In [11]:
# Saves each stock's share of the day's total market value, which is enough to run the analysis notebooks
# z-scores, rankings and market value weights do not change, because only the scale of each day changes
market_value.div(market_value.sum(axis = 1), axis = 0).to_csv(path + "/" + data_folder + "/" + 'relative_market_value.csv.gz')